# SV with exogenous state covariate (volume / money / avg)

批量对指定合约运行带单一外生变量的 SV 模型，输出参数与关键诊断图。

In [ ]:
# 导入依赖，注释使用中文
import numpy as np
import pandas as pd
from pathlib import Path

from sv_toolkit import (
    list_csv_files,
    load_single_file,
    get_contract_symbol_from_path,
    make_timestamped_root,
    save_param_summary,
    prepare_exog_series,
    run_sv_with_exog,
)
from sv_toolkit.plotting import (
    plot_volatility,
    plot_param_posterior,
    plot_standardized_residuals,
    plot_mixture_usage,
)

# 配置路径与随机种子
data_dir = Path('../2005年__20250905')
output_root = make_timestamped_root(Path('outputs'))
experiment_dir = output_root / 'exog_state_tests'
experiment_dir.mkdir(parents=True, exist_ok=True)

print(f'Data dir: {data_dir}')
print(f'Outputs: {experiment_dir}')

## 1. 选择需要处理的合约文件

In [ ]:
# 目标 CSV 文件名单
target_files = [
    'AG_主力合约_1m数据.csv',
    'AL_主力合约_1m数据.csv',
    'AP_主力合约_1m数据.csv',
    'AU_主力合约_1m数据.csv',
    'BB_主力合约_1m数据.csv',
    'BU_主力合约_1m数据.csv',
    'CF_主力合约_1m数据.csv',
    'CJ_主力合约_1m数据.csv',
    'CS_主力合约_1m数据.csv',
    'CU_主力合约_1m数据.csv',
    'CY_主力合约_1m数据.csv',
]

all_csv = list_csv_files(data_dir, max_files=400)
name_to_path = {p.name: p for p in all_csv}
selected_paths = []
for name in target_files:
    if name not in name_to_path:
        print(f"Warning: {name} not found under {data_dir}")
    else:
        selected_paths.append(name_to_path[name])

if not selected_paths:
    raise RuntimeError('None of the target files were found.')

print('Selected contracts:')
for p in selected_paths:
    print(' -', p.name)

## 2. 逐合约 + 逐外生变量运行 SV MCMC

In [ ]:
# 要尝试的外生列
exog_cols = ['volume', 'money', 'avg']

datasets = {}

for file_path in selected_paths:
    symbol = get_contract_symbol_from_path(file_path)
    print(f"\n=== {symbol}: loading data from {file_path.name} ===")

    contract_out_dir = experiment_dir / symbol
    contract_out_dir.mkdir(parents=True, exist_ok=True)

    # 读取数据（子采样 5 分钟，限定时间区间）
    r, y_star, df, _ = load_single_file(
        file_path,
        contract_code=None,
        start_time='2015-01-01',
        end_time='2025-12-31',
        max_rows=None,
        sample_every=5,
        state_exog_col=None,
        log_exog=True,
    )

    if len(r) <= 100 or len(y_star) <= 100:
        print(f"[{symbol}] Not enough data points (T={len(r)}), skip SV MCMC for this contract.")
        continue

    for idx, exog_col in enumerate(exog_cols):
        exog = prepare_exog_series(df, exog_col, log_transform=True, zscore=True)
        if exog is None:
            continue

        print(f"[{symbol}] Running SV with exogenous column: {exog_col}")
        mcmc_results, exog_scaled = run_sv_with_exog(
            r,
            y_star,
            exog,
            n_iter=400,
            burn_in=40,
            thin=2,
            rng_seed=2025 + idx,
            progress_every=20,
            store_s=True,
        )

        tag = f"{symbol}_{exog_col}"
        extra_info = {
            'file_name': file_path.name,
            'contract_tag': symbol,
            'exog_column': exog_col,
            'exog_log': True,
            'exog_zscore': True,
            'T': len(r),
            'n_iter': 400,
            'burn_in': 40,
            'thin': 2,
            'sample_every': 5,
            'start_time': '2015-01-01',
            'end_time': '2025-12-31',
        }
        save_param_summary(mcmc_results, contract_out_dir, tag, extra_info=extra_info)

        # 仅输出与模型差异相关的图
        vol_png = plot_volatility(mcmc_results['h'], df, contract_out_dir, title_suffix=tag)
        param_png = plot_param_posterior(mcmc_results, contract_out_dir, title_suffix=tag)
        std_png = plot_standardized_residuals(r, mcmc_results, contract_out_dir, title_suffix=tag)
        mix_png = None
        if 's' in mcmc_results and len(mcmc_results['s']) > 0:
            mix_png = plot_mixture_usage(mcmc_results['s'], contract_out_dir, title_suffix=tag)

        print(f"[{symbol}] Saved: {vol_png}, {param_png}, {std_png}, mixture={mix_png}")

    datasets[symbol] = {
        'r': r,
        'y_star': y_star,
        'df': df,
    }

print(f"\nFinished exogenous SV runs for {len(datasets)} contracts: {list(datasets.keys())}")